<a href="https://colab.research.google.com/github/denys-babak/data-cleaning/blob/main/%5BNOTEBOOK%5D_1_Data_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Clearance
## Importing tables

In [ ]:
import pandas as pd

In [ ]:
url = "https://drive.google.com/file/d/1Mm-8yZymRUNfOz0ubhHMOGVVV890F6Ws/view?usp=drive_link" # products
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
products_df = pd.read_csv(path)
df_products = products_df.copy()

In [ ]:
url = "https://drive.google.com/file/d/1r_d9-hQflMgkQW4WSVCfqVojD0aVwU5h/view?usp=drive_link" # brands
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
brands_df = pd.read_csv(path)

In [ ]:
url = "https://drive.google.com/file/d/1arSZO57Eax5q4eOH6yIL2FRa8NkU9gPk/view?usp=drive_link" # orders
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orders_df = pd.read_csv(path)

In [ ]:
url = "https://drive.google.com/file/d/1u0HJ8YA7112xc75KL0d9p3O_ieJiJzZe/view?usp=drive_link" # orderlines
path = "https://drive.google.com/uc?export=download&id="+url.split("/")[-2]
orderlines_df = pd.read_csv(path)

**# # # unit_price contains malformed decimal formatting (e.g. 1.137.99)**

In [ ]:
# orderlines_df["unit_price"] = orderlines_df["unit_price"].astype(float)

In [ ]:
df_products.duplicated().sum()

np.int64(8746)

##  **!!! 1. Orderlines !!!**

# **Rename columns**

In [ ]:
df_orderlines = orderlines_df.copy()

df_orderlines = df_orderlines.rename(columns={
"id": "orderline_id",
"id_order": "order_id", # to make orders and orderlines consistent
"date": "order_date"
})


# **Drop product_id column**

In [ ]:
# df_orderlines['product_id'].unique() # check unique values
df_orderlines = df_orderlines.drop('product_id', axis=1) # removed legacy column product_id
df_orderlines


,orderline_id,order_id,product_quantity,sku,unit_price,order_date
0,1119109,299539,1,OTT0133,18.99,2017-01-01 00:07:19
1,1119110,299540,1,LGE0043,399.00,2017-01-01 00:19:45
2,1119111,299541,1,PAR0071,474.05,2017-01-01 00:20:57
3,1119112,299542,1,WDT0315,68.39,2017-01-01 00:51:40
4,1119113,299543,1,JBL0104,23.74,2017-01-01 01:06:38
...,...,...,...,...,...,...
293978,1650199,527398,1,JBL0122,42.99,2018-03-14 13:57:25
293979,1650200,527399,1,PAC0653,141.58,2018-03-14 13:57:34
293980,1650201,527400,2,APP0698,9.99,2018-03-14 13:57:41
293981,1650202,527388,1,BEZ0204,19.99,2018-03-14 13:58:01


In [ ]:
# df_orderlines['unit_price'].mean()
# df_orderlines["unit_price"] = df_orderlines["unit_price"].astype(float)

df_orderlines[df_orderlines["unit_price"].str.count("\.") > 1].head(20).sort_values(by="unit_price")

<>:4: SyntaxWarning: invalid escape sequence '\.'
<>:4: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_2672/2450726015.py:4: SyntaxWarning: invalid escape sequence '\.'
  df_orderlines[df_orderlines["unit_price"].str.count("\.") > 1].head(20).sort_values(by="unit_price")


,orderline_id,order_id,product_quantity,sku,unit_price,order_date
187,1119456,299698,1,APP1634,1.019.00,2017-01-01 13:32:11
6,1119115,299544,1,APP1582,1.137.99,2017-01-01 01:17:21
164,1119412,299679,1,APP1574,1.348.99,2017-01-01 13:00:42
159,1119405,299664,1,APP0958,1.351.99,2017-01-01 12:57:21
155,1119392,299669,1,APP1196,1.426.99,2017-01-01 12:46:06
163,1119411,299678,1,APP1196,1.426.99,2017-01-01 13:00:40
174,1119431,299685,1,APP1617,1.845.99,2017-01-01 13:18:38
131,1119345,299646,1,PAC1567,2.050.99,2017-01-01 12:18:41
105,1119294,299629,1,QNA0175,2.054.99,2017-01-01 11:39:13
111,1119303,299632,1,APP1269,2.172.99,2017-01-01 11:46:18


# **??? Convert unit_price to numeric**

In [ ]:
# df_orderlines["unit_price"] = pd.to_numeric(df_orderlines["unit_price"] # errors="coerce")

In [ ]:
df_merged = df_orderlines[["sku","unit_price"]].merge(
    df_products[["name","desc","sku","price", "promo_price"]],
    on="sku",
    how="inner"
)

df_merged

# 1 weird 199.904 = 19.99
# 2 with two dots 5.689.989 = 568.99

# 1 filter if there is 1 (/.) and 3 numbers after dot >> split to xx.xx
# 2 filter that there are 2 dots >> split to xxx.xx

,sku,unit_price,name,desc,price,promo_price
0,OTT0133,18.99,Otterbox iPhone Case Symmetry 2.0 SE / 5s / 5 ...,resistant cover and thin beveled edges for iPh...,34.99,199.904
1,LGE0043,399.00,"27UD58-B LG Monitor 27 ""4K UHD DisplayPort",Monitor for gamers and multimedia professional...,429,3.989.999
2,PAR0071,474.05,Parrot Bebop 2 White + Command FLYPAD and FPV ...,cuadricóptero wireless remote control with 25 ...,699,5.689.989
3,WDT0315,68.39,"Blue WD 2TB Hard Drive 35 ""Mac and PC",Internal Hard Drive Western Digital 2TB 3.5-in...,79,639.945
4,JBL0104,23.74,Gray Bluetooth Speaker JBL GO,Compact Bluetooth Handsfree for iPhone iPad an...,29.9,27.99
...,...,...,...,...,...,...
378392,JBL0122,42.99,JBL T450 BT Bluetooth Headset Black,Wireless headphones with folding design with 1...,49.95,429.901
378393,PAC0653,141.58,Samsung SSD 850 expansion kit EVO 250GB + Data...,SSD upgrade kit 2008-2010 250 GB MacBook and M...,215.98,1.415.845
378394,APP0698,9.99,Apple Lightning Cable Connector to USB 1m Whit...,Apple Lightning USB Cable 1 meter to charge an...,25,99.898
378395,BEZ0204,19.99,"Be.ez LArobe Case Mix Macbook 12 ""Green",Macbook thin sheath 12 inches.,29.99,199.904


# **Converted order_date to date type**

In [ ]:
df_orderlines["order_date"] = pd.to_datetime(df_orderlines["order_date"])

In [ ]:
df_orderlines['sku']

,sku
0,OTT0133
1,LGE0043
2,PAR0071
3,WDT0315
4,JBL0104
...,...
293978,JBL0122
293979,PAC0653
293980,APP0698
293981,BEZ0204


# **Verify SKUs**





In [ ]:
mask = df_orderlines["sku"].str.match(r"^[A-Z]{3}\d{4}")

mask.value_counts()

df_orderlines[~mask]

,orderline_id,order_id,product_quantity,sku,unit_price,order_date
1210,1121570,300660,1,par0072,199.00,2017-01-02 15:45:30
2852,1128367,302129,1,par0072,199.00,2017-01-03 23:08:10
44108,1210722,337879,1,AP20023,868.99,2017-03-22 17:47:55
44689,1211767,338359,1,AP20021,819.99,2017-03-23 19:59:22
44759,1211888,338414,1,AP20021,819.99,2017-03-23 23:43:21
...,...,...,...,...,...,...
293335,1648987,526781,1,AP20388,319.00,2018-03-14 02:30:11
293344,1648999,526789,1,AP20351,2.999.01,2018-03-14 05:00:22
293415,1649122,526836,1,AP20462,1.649.00,2018-03-14 10:53:04
293619,1649517,527045,1,AP20388,319.00,2018-03-14 11:49:12




```
# This is formatted as code
```

## **!!! 2. Orders !!!**

In [ ]:
df_orders = orders_df.copy()


# **Converted created_date to date type**

In [ ]:
df_orders.rename(columns={"created_date": "order_date_and_time"}, inplace=True)

df_orders["order_date_"] = pd.to_datetime(df_orders["order_date_and_time"]).dt.date # created a new column with a date
df_orders.rename(columns={"order_date_": "order_date"}, inplace=True)

df_orders.columns
df_orders["order_date"] = pd.to_datetime(df_orders["order_date"]) # Converted order_date to date type
df_orders["order_date_and_time"] = pd.to_datetime(df_orders["order_date_and_time"]) # Converted order_date_and_time to date type

In [ ]:
# reordered columns
cols = list(df_orders.columns)

cols.remove("order_date")
cols.insert(2, "order_date")

df_orders = df_orders[cols]



In [ ]:
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             226909 non-null  int64         
 1   order_date_and_time  226909 non-null  datetime64[ns]
 2   order_date           226909 non-null  datetime64[ns]
 3   total_paid           226904 non-null  float64       
 4   state                226909 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


In [ ]:
# download the clean df as csv file
df_orders.to_csv("orders_cleaned.csv", index=False)

from google.colab import files
files.download("orders_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
df_orders.columns
df_orders = df_orders.loc[:, ~df_orders.columns.duplicated()]
df_orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 226909 entries, 0 to 226908
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             226909 non-null  int64         
 1   order_date_and_time  226909 non-null  datetime64[ns]
 2   order_date           226909 non-null  datetime64[ns]
 3   total_paid           226904 non-null  float64       
 4   state                226909 non-null  object        
dtypes: datetime64[ns](2), float64(1), int64(1), object(1)
memory usage: 8.7+ MB


### There were 5 total_paid NaN's dropped**

In [ ]:
df_orders.isna().sum()
df_orders[df_orders['total_paid'].isna()] # 5 total_paid Nans

df_orders = df_orders.dropna(subset=['total_paid']) # drop them

# we also need to remove order items from df_orderlines
orders_to_remove = ['427314', '431655', '447411', '448966', '449596']
mask_to_check_orders = df_orderlines['order_id'].astype(str).isin(orders_to_remove)
df_orderlines = df_orderlines[~mask_to_check_orders]

df_orders[df_orders.isna().any(axis=1)]

df_orders.isna().sum()

,0
order_id,0
order_date_and_time,0
order_date,0
total_paid,0
state,0


## **!!! 3. Products !!!**

In [ ]:
products_cleaned_2 = products_df.copy()

products_cleaned_2['price'].isna().sum()
products_cleaned_2 = products_cleaned_2.dropna(subset=['price']) # drop 46 Nan prices

products_cleaned_2.duplicated().sum()
products_cleaned_2 = products_cleaned_2.drop_duplicates() # drop duplicates - np.int64(8746)

products_cleaned_2["desc"] = products_cleaned_2["desc"].fillna(products_cleaned_2["name"]) # replaced desc > NaN with "name"
products_cleaned_2.isna().sum()


products_cleaned_2[products_cleaned_2["type"].isna()]
products_cleaned_2["type"] = products_cleaned_2["type"].fillna(0) # filled empty types with 0 not to loose any rows
products_cleaned_2.isna().sum()

# function to clen . in a column and rejoin the parts
def clean_price(x):
    x = str(x)

    if x.count(".") > 1:
        parts = x.split(".")
        x = "".join(parts[:-1]) + "." + parts[-1]

    return float(x)


products_cleaned_2["promo_price"] = products_cleaned_2["promo_price"].apply(clean_price) #applied it on the promo_price column

# products_cleaned['price'] = pd.to_numeric(products_cleaned['price'])

In [ ]:
# function to classify data to fix 400+ broken prices to save them

def fix_price_mask(df, column):
    col = df[column].astype(str)

    masks = {
        "int": col.str.count(r"\.") == 0,
        "one_digit": col.str.match(r'^\d{1}\.\d{2}$', na=False),
        "two_digits": col.str.match(r'^\d{2}\.\d{2}$', na=False),
        "three_digits": col.str.match(r'^\d{3}\.\d{2}$', na=False),
        "four_digits": col.str.match(r'^\d{4}\.\d{2}$', na=False),
    }

    return masks

masks = fix_price_mask(products_cleaned_2, "price")

In [ ]:
#apply the function to clean prices and create a new column price_fixed
clean_price = products_cleaned_2['price'].astype(str).str.replace(".", "", regex=False)

products_cleaned_2.loc[masks["int"], "price_fixed"] = products_cleaned_2.loc[masks["int"], "price"]

products_cleaned_2.loc[masks["one_digit"], "price_fixed"] = products_cleaned_2.loc[masks["one_digit"], "price"]

products_cleaned_2.loc[masks["two_digits"], "price_fixed"] = (
    clean_price[masks["two_digits"]].str[:2] + "." +
    clean_price[masks["two_digits"]].str[2:4]
)

products_cleaned_2.loc[masks["three_digits"], "price_fixed"] = (
    clean_price[masks["three_digits"]].str[:3] + "." +
    clean_price[masks["three_digits"]].str[3:5]
)

products_cleaned_2.loc[masks["four_digits"], "price_fixed"] = (
    clean_price[masks["four_digits"]].str[:4] + "." +
    clean_price[masks["four_digits"]].str[4:6]
)

products_cleaned_2['price_fixed'] = pd.to_numeric(products_cleaned_2['price_fixed']) # set price_fixed to numeric

In [ ]:
#repeatedly detects and flags suspicious promo prices compared to price_fixed
for _ in range(10):
    mask = (
    products_cleaned_2["promo_price"].notna()
    & products_cleaned_2["price_fixed"].notna()
    & (products_cleaned_2["price_fixed"] > 0)
    & (products_cleaned_2["promo_price"] > products_cleaned_2["price_fixed"] * 3)
)

    if not mask.any():
        break

    products_cleaned_2.loc[mask, "promo_price"] /= 10

# divide by 10 to fix:
#wrong decimal placement
#data import errors (e.g. 1999 instead of 199.9)
#unit conversion mistakes

# final formatting
products_cleaned_2["promo_price"] = products_cleaned_2["promo_price"].round(2) # rounds values to 2 decimals (currency formatting)

In [ ]:
products_cleaned_2 = products_cleaned_2.drop(columns=["price"]) # drop broken column
products_cleaned_2 = products_cleaned_2.rename(columns={"price_fixed": "price"}) # rename a fixed column

# reorder the new column to be near promo price
cols = list(products_cleaned_2.columns)

cols.remove("price")
cols.insert(4, "price")

products_cleaned_2 = products_cleaned_2[cols]


In [ ]:
products_cleaned_2['price'].isna().sum() # drop Nans in the price column
products_cleaned_2 = products_cleaned_2.dropna(subset=['price'])

In [ ]:
# download the clean df as csv file
products_cleaned_2.to_csv("products_cleaned.csv", index=False)

from google.colab import files
files.download("products_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# DRAFT!!! Maybe will use in the future .. Converting "promo_price" to the same form as "unit_price"


After merging tables, removing first dots and astype-float

In [ ]:
def clean_price(x):
    x = str(x)

    if x.count(".") > 1:
        parts = x.split(".")
        x = "".join(parts[:-1]) + "." + parts[-1]

    return float(x)

In [ ]:
products_df_cl["promo_price"] = products_df_cl["promo_price"].apply(clean_price)
products_df_cl[["name","price","promo_price"]]

products_df_cl.duplicated().sum()
products_df_cl = products_df_cl.drop_duplicates()

NameError: name 'products_df_cl' is not defined

In [ ]:
df_orderlines[df_orderlines["unit_price"].str.count("\.") > 1].sort_values(by = "unit_price")
df_merged = df_orderlines.merge(
    df_products,
    on="sku"
)

In [ ]:
# clean raw strings first
df_merged["unit_price"] = df_merged["unit_price"].apply(clean_price)
df_merged["promo_price"] = df_merged["promo_price"].apply(clean_price)

# fix scaling issues
for _ in range(10):
    mask = (
        df_merged["promo_price"].notna()
        & df_merged["unit_price"].notna()
        & (df_merged["unit_price"] > 0)
        & (df_merged["promo_price"] > df_merged["unit_price"] * 3)
    )

    if not mask.any():
        break

    df_merged.loc[mask, "promo_price"] /= 10

# final formatting
df_merged["promo_price"] = df_merged["promo_price"].round(2)

In [ ]:
products_df_cl.duplicated().sum()



In [ ]:
for _ in range(10):
    mask = (
        products_df_cl["promo_price"].notna()
        & products_df_cl["price"].notna()
        & (products_df_cl["price"] > 0)
        & (products_df_cl["promo_price"] > products_df_cl["price"] * 3)
    )

    if not mask.any():
        break

    products_df_cl.loc[mask, "promo_price"] /= 10

products_df_cl["promo_price"] = products_df_cl["promo_price"].round(2)

In [ ]:
df_merged.to_csv("clean_prices.csv", index=False)

from google.colab import files
files.download("clean_prices.csv")

## **!!! 4. Orderlines !!!**

In [ ]:
url = 'https://drive.google.com/file/d/1cje_pWKwk-eu-UJaW35C6wVp2jypPOhG/view?usp=sharing'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
products_fixed = pd.read_csv(path)

url = 'https://drive.google.com/file/d/1u0HJ8YA7112xc75KL0d9p3O_ieJiJzZe/view?usp=drive_link'
path = 'https://drive.google.com/uc?export=download&id='+url.split('/')[-2]
orderlines = pd.read_csv(path)

# orderlines 293983 rows × 7 columns

In [ ]:
df_orderlines = orderlines.copy()
df_products = products_fixed.copy()

In [ ]:
# merge for checking unknown products and comparing promo_price, price, unit_price

df_merged = df_orderlines.merge(
    df_products[["sku", "name", "price", "promo_price"]],
    on="sku",
    how="left"
)

missing_products = df_merged["name"].isna() # get the order ids of the orders where the products is unknown and should be removed

order_ids_to_remove = df_merged.loc[missing_products, "id_order"].unique()

df_orderlines_clean = df_merged[
    ~df_merged["id_order"].isin(order_ids_to_remove)
]

df_orderlines_clean

In [ ]:
# renamed columns
df_orderlines_clean = df_orderlines_clean.rename(columns={
"id": "orderline_id",
"id_order": "order_id",
"date": "order_date"
})

#converted order_date to datetime
df_orderlines_clean['order_date'] = pd.to_datetime(df_orderlines_clean['order_date'])

In [ ]:
df_orderlines_clean_unit_price = df_orderlines_clean.copy()

df_orderlines_clean_unit_price

In [ ]:
def clean_price(x):
    x = str(x)

    if x.count(".") > 1:
        parts = x.split(".")
        x = "".join(parts[:-1]) + "." + parts[-1]

    return float(x)

In [ ]:
df_orderlines_clean["unit_price"] = df_orderlines_clean["unit_price"].apply(clean_price)

for _ in range(10):
    mask = (
        df_orderlines_clean["promo_price"].notna()
        & df_orderlines_clean["unit_price"].notna()
        & (df_orderlines_clean["unit_price"] > 0)
        & (df_orderlines_clean["promo_price"] > df_orderlines_clean["unit_price"] * 3)
    )

    if not mask.any():
        break

    df_orderlines_clean.loc[mask, "promo_price"] /= 10

# final formatting
df_orderlines_clean["promo_price"] = df_orderlines_clean["promo_price"].round(2)

In [ ]:
df_orderlines_clean['unit_price'] = pd.to_numeric(df_orderlines_clean['unit_price'])
df_orderlines_clean

In [ ]:
df_merged_test.sort_values('unit_price').head(20)

drop_orderlines = df_orderlines_clean['unit_price'] <= 0 #drop 0 or negative price
df_orderlines_clean = df_orderlines_clean[~drop_orderlines]



In [ ]:
df_orderlines_clean = df_orderlines_clean.drop(columns=["price", "promo_price"])
df_orderlines_clean.info()

df_orderlines_clean["order_date_only"] = pd.to_datetime(df_orderlines_clean["order_date"]).dt.date
df_orderlines_clean



In [ ]:
df_orderlines_clean.drop(columns='product_id', inplace=True)
df_orderlines_clean

In [ ]:
df_orderlines_clean.to_csv("clean_orderlines.csv", index=False)

from google.colab import files
files.download("clean_orderlines.csv")